# numcompute_stream — Streaming ML Demo

This notebook demonstrates the streaming workflow end to end:

1. **Generate / load** a dataset and write it to CSV.
2. **Stream** it in chunks with `io.load_csv_chunked`.
3. **Incrementally train** a `Pipeline` (scaler + Random Forest) with `partial_fit`, one chunk at a time.
4. **Log** per-chunk metrics with `StreamTrainer` (prequential test-then-train).
5. **Visualise** accuracy over time, compare a single tree vs the forest, and show a confusion matrix.

Everything is built on NumPy + matplotlib only.

In [1]:
import sys, os
# Make the package importable when running from the repo root or demo/.
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt

from numcompute_stream import io
from numcompute_stream.preprocessing import StandardScaler
from numcompute_stream.ensemble import RandomForestClassifier
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.utils import iter_chunks
from numcompute_stream import visualise

np.set_printoptions(precision=3, suppress=True)

## 1. Create a dataset and save it as CSV

We synthesise a 3-class problem with some overlap so the metrics are interesting (not a trivial 1.0 everywhere), then persist it to disk so the rest of the demo reads it back as a *stream*.

In [2]:
rng = np.random.default_rng(42)
n_per_class, n_features, n_classes = 1500, 6, 3
centres = rng.normal(0, 3.0, (n_classes, n_features))
X = np.vstack([rng.normal(centres[c], 2.2, (n_per_class, n_features))
               for c in range(n_classes)])
y = np.concatenate([np.full(n_per_class, c) for c in range(n_classes)])
perm = rng.permutation(len(y)); X, y = X[perm], y[perm]

# Persist as a single CSV: features then label in the last column.
data = np.column_stack([X, y])
io.save_csv(data, 'stream_data.csv')
print('Wrote stream_data.csv with shape', data.shape)

Wrote stream_data.csv with shape (4500, 7)


## 2. Stream the CSV in chunks

`io.load_csv_chunked` is a generator: it never loads the whole file into memory. Each chunk is split back into `X_chunk` / `y_chunk`.

In [3]:
CHUNK = 250

def stream_chunks(path, chunk_size):
    for block in io.load_csv_chunked(path, chunksize=chunk_size):
        yield block[:, :-1], block[:, -1].astype(int)

# Peek at the first chunk.
first_X, first_y = next(stream_chunks('stream_data.csv', CHUNK))
print('First chunk:', first_X.shape, first_y.shape, 'labels:', np.unique(first_y))

First chunk: (250, 6) (250,) labels: [0 1 2]


## 3. Incrementally train a Pipeline with `partial_fit`

The pipeline scales features online (running mean/variance via Welford) and feeds them to a streaming Random Forest. We drive it with `StreamTrainer`, which does **prequential** evaluation: each chunk is predicted *before* being learned from, so the logged accuracy is an honest estimate of generalisation on unseen data.

In [4]:
classes = np.array([0, 1, 2])

forest_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=15, max_depth=8, random_state=0)),
])

forest_trainer = StreamTrainer(forest_pipe, prequential=True, verbose=True)
forest_hist = forest_trainer.run(stream_chunks('stream_data.csv', CHUNK), classes=classes)
print('\nFinal cumulative accuracy:', round(forest_hist['cumulative_accuracy'][-1], 3))

[chunk   0] n= 250  acc=0.996  cum_acc=0.996  t=426.6ms  mem=0.21MB


[chunk   1] n= 250  acc=0.928  cum_acc=0.962  t=983.8ms  mem=0.41MB


[chunk   2] n= 250  acc=0.956  cum_acc=0.960  t=1643.2ms  mem=0.62MB


[chunk   3] n= 250  acc=0.924  cum_acc=0.951  t=2122.4ms  mem=0.82MB


[chunk   4] n= 250  acc=0.952  cum_acc=0.951  t=2670.7ms  mem=1.02MB


...

[chunk  13] n= 250  acc=0.956  cum_acc=0.950  t=8953.2ms  mem=2.83MB


[chunk  14] n= 250  acc=0.948  cum_acc=0.950  t=9570.8ms  mem=3.04MB


[chunk  15] n= 250  acc=0.952  cum_acc=0.950  t=9828.6ms  mem=3.24MB


[chunk  16] n= 250  acc=0.944  cum_acc=0.950  t=10573.0ms  mem=3.44MB


[chunk  17] n= 250  acc=0.944  cum_acc=0.949  t=10903.3ms  mem=3.64MB

Final cumulative accuracy: 0.949


## 4. Visualise metrics over time

`visualise.plot_metric_over_time` plots any logged per-chunk metric. Here we show prequential accuracy and the macro-F1 score as the stream progresses.

In [5]:
visualise.plot_metric_over_time(
    forest_hist['accuracy'],
    title='Random Forest — prequential accuracy over the stream',
    ylabel='Accuracy', show=True)

visualise.plot_metric_over_time(
    forest_hist['f1'],
    title='Random Forest — macro F1 over the stream',
    ylabel='F1', show=True)

<Figure size 800x450 with 1 Axes>

## 5. Compare a single tree vs the forest

We run a single `DecisionTreeClassifier` over the identical stream and overlay the two accuracy curves. The forest is typically steadier (lower variance) because soft-voting averages out individual trees.

In [6]:
tree_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('tree', DecisionTreeClassifier(max_depth=8, random_state=0)),
])
tree_trainer = StreamTrainer(tree_pipe, prequential=True, verbose=False)
tree_hist = tree_trainer.run(stream_chunks('stream_data.csv', CHUNK), classes=classes)

visualise.compare_models(
    tree_hist['accuracy'], forest_hist['accuracy'],
    labels=('Single tree', 'Random forest'),
    title='Single tree vs Random forest — prequential accuracy',
    ylabel='Accuracy', show=True)

print('Single tree   final cum_acc:', round(tree_hist['cumulative_accuracy'][-1], 3))
print('Random forest final cum_acc:', round(forest_hist['cumulative_accuracy'][-1], 3))

Single tree   final cum_acc: 0.934
Random forest final cum_acc: 0.949


## 6. Confusion matrix on a held-out chunk

Finally we score the trained forest on a fresh chunk and render the confusion matrix to see *where* errors land between the three classes.

In [7]:
# Generate a fresh held-out batch from the same distribution.
Xh = np.vstack([rng.normal(centres[c], 2.2, (200, n_features)) for c in range(n_classes)])
yh = np.concatenate([np.full(200, c) for c in range(n_classes)])
ph = rng.permutation(len(yh)); Xh, yh = Xh[ph], yh[ph]

y_pred = forest_pipe.predict(Xh)
visualise.plot_predictions_vs_ground_truth(
    yh, y_pred, title='Forest — confusion matrix on held-out data', show=True)

<Figure size 600x500 with 2 Axes>

## Summary

- Data was consumed as a **stream** of CSV chunks, never fully in memory.
- A scaler + Random Forest were trained **incrementally** via `partial_fit`.
- `StreamTrainer` logged **prequential** accuracy / precision / recall / F1, per-chunk latency and model size.
- `visualise` produced accuracy-over-time, model-comparison and confusion-matrix plots.

See `benchmark/benchmark_stream.py` for a performance comparison of the single tree vs the forest, and of vectorised vs loop-based class counting.